## Wi-Fi Sensing
### Imports de bibliotecas

In [ ]:
# pandas for data manipulation
import pandas as pd

# numpy for numerical operations
import numpy as np

# pyplot for plotting
import matplotlib.pyplot as plt

# re for regular expressions
import re

# sklearn for machine learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, make_scorer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# joblib for saving/loading models
import joblib

# other .py files
from utils.csv_import import get_csv_files

# seaborn for advanced plotting
import seaborn as sns

from typing import Dict

### Retrieve CSV files

In [ ]:
path: str = "C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project\\CSI DATA RENAMED [alterado]"
FileMap = Dict[str, Dict[str, str]]

csv_loureiro: FileMap = {}
csv_diana: FileMap = {}
csv_afinar: FileMap = {}

csv_loureiro, csv_diana, csv_afinar = get_csv_files(path)

In [ ]:
def print_useful_info(file: FileMap) -> None:
	
	# Print information about the files dictionaries
	print("Número de ficheiros recolhidos em ", file, ":", len(file))
	
    # avoid StopIteration if dict is empty
	esp_count = len(next(iter(file.values()))) if file else 0
	print("ESPs por cenário:", esp_count)

	# Print all scenarios names
	print("\nTodos os cenários:", sorted(file.keys()))

	# Count scenarios by letter prefix
	scenario_prefixes: Dict[str, int] = {}
	for scenario in file:
		prefix = scenario[0]
		scenario_prefixes[prefix] = scenario_prefixes.get(prefix, 0) + 1

	print("\nScenarios count by prefix:")
	for prefix, count in sorted(scenario_prefixes.items()):
		print(f"Prefix {prefix}: {count} scenarios")

	print("\n")

In [ ]:
print_useful_info(csv_loureiro)
print_useful_info(csv_diana)
print_useful_info(csv_afinar)

### Process ficheiros CSV

#### Tabela magnitudes

In [ ]:
# função que processo o arquivo CSV completo
# # e extrai a info da coluna do CSI
# # converte em números complexos
# # calcula a magnitude (2.º passo)
# # 
# e faz ainda uma 1a limpeza
# # remove 2 primeiras subportadoras
# # remove colunas a zero
# # aplica FFT shift

# da tese
# # Vector size verification
# # Subcarrier layout and magnitude computation

csiMap = Dict[str, Dict[str, np.ndarray]]

def process_csi(file: str):
    df = pd.read_csv(file, header=None)
    csi_col = df.iloc[:, 26]

    valid_csi = []
    for entry in csi_col:
        match = re.search(r'\[(.*?)\]', str(entry))
        if not match:
            continue
        nums = [float(n) for n in re.findall(r'-?\d+', match.group(1))]
        if len(nums) == 128:
            valid_csi.append(nums)

    valid_csi = np.array(valid_csi)
    complex_csi = valid_csi[:, ::2] + 1j * valid_csi[:, 1::2]
    magnitudes = np.abs(complex_csi)

    # Limpeza: remover 2 primeiras subportadoras e colunas todas-zero
    magnitudes = magnitudes[:, 2:]
    magnitudes = magnitudes[:, ~np.all(magnitudes == 0, axis=0)]
    magnitudes = np.fft.fftshift(magnitudes, axes=1)

    return magnitudes

# cria o dict magnitudes
# itera sobre cada posicao da grelha
# # para cada posicao, itera sobre cada esp_id e path
# # devolve o dict magnitudes preenchido
def process_magnitudes(file: FileMap) -> Dict[str, Dict[str, np.ndarray]]:
    
    magnitudes: Dict[str, Dict[str, np.ndarray]] = {}

    for posicao in file:

        magnitudes[posicao] = {}

        # dict com "esp_id" e "path"
        esps = file[posicao]

        for esp_id, ficheiro in esps.items():
            magnitudes[posicao][esp_id] = process_csi(ficheiro)
            
    return magnitudes

# o dict magnitudes fica
# posicao1
# # esp1
# # # array de arrays (cada leitura é um array)
# # ...
# # esp4
# # # array de arrays (cada leitura é um array)

In [ ]:
magnitudes_loureiro: csiMap = {}
magnitudes_diana: csiMap = {}
magnitudes_afinadas: csiMap = {}

magnitudes_loureiro = process_magnitudes(csv_loureiro)
magnitudes_diana = process_magnitudes(csv_diana)
magnitudes_afinadas = process_magnitudes(csv_afinar)

#### Cálculo da média

In [ ]:
# cálculo das médias das subportadoras

# função que extrai valores de determiada subportadora
# # recolhe apenas uma fracção dos valores (1/6 * nº leituras)
# # e devolve um dicionário com os resultados
# # # dict esp: array de valores
def extrai_valores_sc(magnitudes_dict: csiMap, posicao: str, subcarrier: int, frac: float) -> Dict[str, np.ndarray]:
    resultados: Dict[str, np.ndarray] = {}
    n: int
    for esp, matriz in magnitudes_dict[posicao].items():
        n = int(matriz.shape[0] * frac)
        resultados[esp] = matriz[:n, subcarrier]
    return resultados

# função que calcula a média dos valores extraídos na função de cima
# retorna um dict com 
# # esp: média
# Nota: np.mean retorna um np.floating; para satisfazer o tipo Dict[str, float]
# convertemos explicitamente para float()
def calcular_media_sc(subportadora_dict: Dict[str, np.ndarray]) -> Dict[str, float]:
    return {esp: float(np.mean(valores)) for esp, valores in subportadora_dict.items()}

# função que calcula a média para cada subportadora
# # para cada sc
# # extrai os (n * frac) primeiros valores de cada esp
# # calcula a sua média 
# # # devolve um dict com
# # # subcarrier: {esp_y: média}
def calcular_medias_para_varias_subportadoras(magnitudes_dict: csiMap, posicao: str, subcarrier_indices: list[int], frac: float) -> Dict[int, Dict[str, float]]:
    medias_por_subc: Dict[int, Dict[str, float]] = {}
    for sc in subcarrier_indices:
        valores = extrai_valores_sc(magnitudes_dict, posicao, sc, frac)
        medias = calcular_media_sc(valores)
        medias_por_subc[sc] = medias
    return medias_por_subc

In [ ]:
# Grupo: c06
#   esp: (nº leituras, nº subportadoras)
#   esp1: (76, 51) # 76 é matriz.shape[0]
#   esp2: (71, 51)
#   esp3: (79, 51)
#   esp4: (99, 51)

# posição para retirar normalizações - escolha ARBITRÁRIA (explorar outras opções?)
posicao: str = 'c06'

# inicializa lista de subportadoras e números de ESPs
subcarriers: list[int] = list(range(51))
esps_id: list[int] = [1, 2, 3, 4]

# fração dos valores a considerar - porquê este valor?
frac: float = 1/6

medias_c06: Dict[int, Dict[str, float]] = {}
medias_c06 = calcular_medias_para_varias_subportadoras(magnitudes_loureiro, posicao, subcarriers, frac)
# para a posição c06
# # para a subcarrier_x
# # # esp1: média
# # # esp2: média
# # # esp3: média
# # # esp4: média

# Gera o dicionário final no formato usado no Raspberry Pi
# normalization_means_full = {
#     esp: {
#         subc: medias_c06[subc][f"esp{esp}"]
#         for subc in range(51)
#     }
#     for esp in esp_id
# }
#joblib.dump(normalization_means_full, 'normalization_means_full.pkl')

#### Tabela dados

Criar dataset "dados" (sc x posição) (array)

Ao invés de "magnitudes" (posição x esp) (array de arrays)


In [ ]:
# Parâmetros

# tamanhos fixos selecionados para quando 
# se fizer o "step" ficar número certo de amostras
# 
# sem pessoa tem mais dados  
N = 90
M = 350
O = 60
A = 292
I = 200

tamanhos_loureiro: Dict[str, int] = {}
tamanhos_diana: Dict[str, int] = {}
tamanhos_afinar: Dict[str, int] = {}

tamanhos = {
    'vazio_1': M, 'vazio_2': M,
    'a00': N, 'c05': N, 'a01': N, 'c06': N,
    'a09': N, 'c07': N, 'a10': N, 'c08': N,
    'a11': N, 'c09': N, 'b00': N, 'c10': N,
    'b01': N, 'c11': N, 'b02': N, 'd01': N,
    'b03': N, 'd02': N, 'b04': N, 'd03': N,
    'b05': N, 'd04': N, 'b06': N, 'd05': N,
    'b07': N, 'd06': N, 'b08': N, 'd07': N,
    'b09': N, 'd08': N, 'b10': N, 'd09': N,
    'b11': N, 'd10': N, 'c01': N, 'd11': N,
    'c02': N, 'e04': N, 'c03': N, 'e05': N,
    'c04': N, 'e06': N,
}

tamanhos_diana = {
    'vazio_1': M, 'vazio_2': M,
    'a00': O, 'c05': O, 'a01': O, 'c06': O,
    'a09': O, 'c07': O, 'a10': O, 'c08': O,
    'a11': O, 'c09': O, 'b00': O, 'c10': O,
    'b01': O, 'c11': O, 'b02': O, 'd01': O,
    'b03': O, 'd02': O, 'b04': O, 'd03': O,
    'b05': O, 'd04': O, 'b06': O, 'd05': O,
    'b07': O, 'd06': O, 'b08': O, 'd07': O,
    'b09': O, 'd08': O, 'b10': O, 'd09': O,
    'b11': O, 'd10': O, 'c01': O, 'd11': O,
    'c02': O, 'e04': O, 'c03': O, 'e05': O,
    'c04': O, 'e06': O,
}

tamanhos_afinar = {
    'com_1': A, 'sem_1': A, 'com_2': I, 'sem_2': I,
}

posicoes: list[str] = []
posicoes_afinar: list[str] = []

# do Miguel não há a02
posicoes = ['vazio_1', 'vazio_2', 'a00', 'a01', 'a09', 'a10', 'a11', 'b00', 'b01', 'b02',
        'b03', 'b04', 'b05', 'b06', 'b07', 'b08', 'b09', 'b10', 'b11', 'c01', 'c02', 'c03',
        'c04', 'c05', 'c06', 'c07', 'c08', 'c09', 'c10', 'c11', 'd01', 'd02', 'd03',
        'd04', 'd05', 'd06', 'd07', 'd08', 'd09', 'd10', 'd11', 'e04', 'e05', 'e06']

posicoes_afinar = ['sem_1', 'sem_2', 'com_1', 'com_2']

In [ ]:
# para cada posição
# # ver o tamanho de cada (auto definido em cima)
# # e para cada subportadora
# # # e para cada esp
# # # # extrair o vetor correto
# # # # guardar no dicionário

dadosMap = Dict[int, Dict[str, np.ndarray]]

def criar_dataset(posicoes: list[str], tamanhos: Dict[str, int], magnitudes: csiMap) -> dadosMap:
    
    dados: dadosMap = {}
    dados = {sc: {} for sc in subcarriers}
    
    for posicao in posicoes:
        tamanho = tamanhos[posicao]
        for sc in subcarriers:
            for esp_id in esps_id:
                # acede ao array de arrays
                matriz = magnitudes[posicao][f'esp_{esp_id}']
                # extrai o vetor coluna
                # com (linhas/leituras = tamanho)
                # com (coluna = subc)
                vetor = matriz[:tamanho, sc]
                #print("nº linhas = ", matriz.shape[0], " e tamanho = ", tamanho)
                chave = f"{posicao}_esp_{esp_id}"
                dados[sc][chave] = vetor

    return dados

In [ ]:
dados_loureiro: dadosMap = {}
dados_diana: dadosMap = {}
dados_afinar: dadosMap = {}

dados_loureiro = criar_dataset(posicoes, tamanhos, magnitudes_loureiro)
dados_diana = criar_dataset(posicoes, tamanhos_diana, magnitudes_diana)
dados_afinar = criar_dataset(posicoes_afinar, tamanhos_afinar, magnitudes_afinadas)

In [ ]:
print(len(dados_loureiro[35]["vazio_2_esp_1"]))
print(len(dados_loureiro[35]["a00_esp_1"]))

#### Tabela normalizados

Normalização de dados

In [ ]:
# Função para normalizar
# a função recebe o dicionário de vetores e o dicionário de médias
# e devolve o dicionário de vetores normalizados
# cada vetor é normalizado subtraindo a média e dividindo pela média
def normalizar_vetores(dados: dadosMap, medias: Dict[int, Dict[str, float]]) -> dadosMap:

    normalizados: dadosMap = {}
    normalizados = {sc: {} for sc in subcarriers}

    for sc in subcarriers:
        for posicao in posicoes:
            for esp_id in esps_id:
                
                chave = f"{posicao}_esp_{esp_id}"
                vetor = dados[sc][chave]

                esp = f"esp_{esp_id}"
                media = medias[sc][esp]

                normalizados[sc][chave] = (vetor - media) / media
    return normalizados

# Função para normalizar dados a afinar
def normalizar_vetores_afinar(vetores: dadosMap, medias: Dict[int, Dict[str, float]]) -> dadosMap:

    normalizados: dadosMap = {}
    normalizados = {sc: {} for sc in subcarriers}

    for sc in subcarriers:
        for posicao in posicoes_afinar:
            for esp_id in esps_id:

                chave = f"{posicao}_esp_{esp_id}"
                vetor = vetores[sc][chave]
                
                esp = f"esp_{esp_id}"
                media = medias[sc][esp]

                normalizados[sc][chave] = (vetor - media) / media
    return normalizados

In [ ]:
normalizados_loureiro: dadosMap = {}
normalizados_diana: dadosMap = {}
normalizados_afinar: dadosMap = {}

normalizados_loureiro = normalizar_vetores(dados_loureiro, medias_c06)
normalizados_diana = normalizar_vetores(dados_diana, medias_c06)
normalizados_afinar = normalizar_vetores_afinar(dados_afinar, medias_c06)


In [ ]:
print(len(normalizados_loureiro[34]["vazio_1_esp_1"]))
print(len(normalizados_loureiro[0]["a00_esp_1"]))

#### Cálculo mean, std, max

Média, desvio padrão e valor máximo

In [ ]:
# função para calcular a média em janelas sobrepostas
def media_grupos_overlap(vetor: np.ndarray, tamanho_janela: int, step: int) -> np.ndarray:
    medias: list[float] = []
    for i in range(0, len(vetor) - tamanho_janela + 1, step):
        janela = vetor[i:i + tamanho_janela]
        medias.append(float(np.mean(janela)))
    return np.array(medias)

# função para calcular o desvio padrão em janelas sobrepostas
def std_grupos_overlap(vetor: np.ndarray, tamanho_janela: int, step: int) -> np.ndarray:
    desvios: list[float] = []
    for i in range(0, len(vetor) - tamanho_janela + 1, step):
        janela = vetor[i:i + tamanho_janela]
        desvios.append(float(np.std(janela)))
    return np.array(desvios)

# função para calcular o máximo em janelas sobrepostas,
# evitando repetições dos últimos 3 máximos escolhidos
def max_grupos_overlap(vetor: np.ndarray, tamanho_janela: int, step: int) -> np.ndarray:
    maximos: list[float] = []
    for i in range(0, len(vetor) - tamanho_janela + 1, step):
        janela = vetor[i:i + tamanho_janela]
        candidatos = np.sort(np.unique(janela))[::-1]  # do maior para o menor

        # Exclui os 3 últimos valores
        ultimos = set(maximos[-3:])

        escolhido = None
        for val in candidatos:
            if val not in ultimos:
                escolhido = val
                break

        # Se todos os valores estão nos últimos 3, aceita o maior mesmo assim
        if escolhido is None:
            escolhido = candidatos[0]

        maximos.append(escolhido)

    return np.array(maximos)

In [ ]:
def media_fixed(vetor: np.ndarray, tamanho_janela: int, W: int) -> np.ndarray:
    step = (len(vetor) - tamanho_janela) // (W - 1)
    return media_grupos_overlap(vetor, tamanho_janela, step)[:W]

def std_fixed(vetor: np.ndarray, tamanho_janela: int, W: int) -> np.ndarray:
    step = (len(vetor) - tamanho_janela) // (W - 1)
    return std_grupos_overlap(vetor, tamanho_janela, step)[:W]

def max_fixed(vetor: np.ndarray, tamanho_janela: int, W: int) -> np.ndarray:
    step = (len(vetor) - tamanho_janela) // (W - 1)
    return max_grupos_overlap(vetor, tamanho_janela, step)[:W]

In [ ]:
tamanho_janela: int = 10
step: int = 3
W: int = 27   # nº de janelas fixas para "com pessoa"
G: int = 60   # nº de janelas fixas para "dados_afinar"

media_loureiro: dadosMap = {sc: {} for sc in subcarriers}
media_diana: dadosMap    = {sc: {} for sc in subcarriers}
media_afinar: dadosMap   = {sc: {} for sc in subcarriers}

std_loureiro: dadosMap = {sc: {} for sc in subcarriers}
std_diana: dadosMap    = {sc: {} for sc in subcarriers}
std_afinar: dadosMap   = {sc: {} for sc in subcarriers}

max_loureiro: dadosMap = {sc: {} for sc in subcarriers}
max_diana: dadosMap    = {sc: {} for sc in subcarriers}
max_afinar: dadosMap   = {sc: {} for sc in subcarriers}

for posicao in posicoes:
    for esp_id in esps_id:
        chave = f"{posicao}_esp_{esp_id}"
        for sc in subcarriers:

            valores_l = normalizados_loureiro[sc][chave]
            valores_d = normalizados_diana[sc][chave]

            if posicao.startswith("vazio"):
                # mantém todas as janelas possíveis, sem truncar
                media_l = media_grupos_overlap(valores_l, tamanho_janela, step)
                std_l = std_grupos_overlap(valores_l, tamanho_janela, step)
                m_l = max_grupos_overlap(valores_l, tamanho_janela, step)

                media_d = media_grupos_overlap(valores_d, tamanho_janela, step)
                std_d = std_grupos_overlap(valores_d, tamanho_janela, step)
                m_d = max_grupos_overlap(valores_d, tamanho_janela, step)
                
            else:
                # força W janelas idênticas
                media_l = media_fixed(valores_l, tamanho_janela, W)
                std_l = std_fixed(valores_l, tamanho_janela, W)
                m_l = max_fixed(valores_l, tamanho_janela, W)

                media_d = media_fixed(valores_d, tamanho_janela, W)
                std_d = std_fixed(valores_d, tamanho_janela, W)
                m_d = max_fixed(valores_d, tamanho_janela, W)

            media_loureiro[sc][chave] = media_l
            std_loureiro  [sc][chave] = std_l
            max_loureiro  [sc][chave] = m_l

            media_diana [sc][chave] = media_d
            std_diana   [sc][chave] = std_d
            max_diana   [sc][chave] = m_d

for posicao in posicoes_afinar:
    for esp_id in esps_id:
        chave = f"{posicao}_esp_{esp_id}"
        for sc in subcarriers:
            v = normalizados_afinar[sc][chave]
            media = media_fixed(v, tamanho_janela, G)
            std = std_fixed(v, tamanho_janela, G)
            max = max_fixed(v, tamanho_janela, G)

            media_afinar[sc][chave] = media
            std_afinar[sc][chave] = std
            max_afinar[sc][chave] = max

In [ ]:
print(len(media_loureiro[34]["vazio_1_esp_1"]))
print(len(media_loureiro[0]["a00_esp_1"]))

#### Tabela colunas

In [ ]:
subcarriers_map: dict[int, int] = {i + 1: sc for i, sc in enumerate(subcarriers)}

# dicionários para guardar cada coluna
coluna_media: dict[str, np.ndarray] = {}
coluna_std: dict[str, np.ndarray] = {}
coluna_max: dict[str, np.ndarray] = {}

# dicionários para guardar cada coluna
coluna_media_afinar: dict[str, np.ndarray] = {}
coluna_std_afinar: dict[str, np.ndarray] = {}
coluna_max_afinar: dict[str, np.ndarray] = {}

for sc_label, sc in subcarriers_map.items():
    for esp_id in esps_id:
        # nome da coluna
        name_mean = f"Mean sc_{sc_label} esp_{esp_id}"
        name_std  = f"Std sc_{sc_label} esp_{esp_id}"
        name_max  = f"Max sc_{sc_label} esp_{esp_id}"

        sequencia_mean, sequencia_std, sequencia_max = [], [], []

        # 1) SEM PESSOA: primeiro Loureiro, depois Diana
        sequencia_mean.append(media_loureiro[sc][f"vazio_1_esp_{esp_id}"])
        sequencia_mean.append(media_loureiro[sc][f"vazio_2_esp_{esp_id}"])

        sequencia_mean.append(media_diana[sc][f"vazio_1_esp_{esp_id}"])
        sequencia_mean.append(media_diana[sc][f"vazio_2_esp_{esp_id}"])

        sequencia_std.append(std_loureiro[sc][f"vazio_1_esp_{esp_id}"])
        sequencia_std.append(std_loureiro[sc][f"vazio_2_esp_{esp_id}"])

        sequencia_std.append(std_diana[sc][f"vazio_1_esp_{esp_id}"])
        sequencia_std.append(std_diana[sc][f"vazio_2_esp_{esp_id}"])

        sequencia_max.append(max_loureiro[sc][f"vazio_1_esp_{esp_id}"])
        sequencia_max.append(max_loureiro[sc][f"vazio_2_esp_{esp_id}"])

        sequencia_max.append(max_diana[sc][f"vazio_1_esp_{esp_id}"])
        sequencia_max.append(max_diana[sc][f"vazio_2_esp_{esp_id}"])


        # 2) COM PESSOA: percorre todas as outras posições

        # GRALHA CÓDIGO ORIGINAL
        # # o "if" apenas serve para o grupo "vazio_1" e não para "vazio_2"
        for posicao in posicoes:
            if posicao.startswith("vazio"):
                continue

            key = f"{posicao}_esp_{esp_id}"
            sequencia_mean.append(media_loureiro[sc][key])
            sequencia_mean.append(media_diana[sc][key])

            sequencia_std.append(std_loureiro[sc][key])
            sequencia_std.append(std_diana[sc][key])

            sequencia_max.append(max_loureiro[sc][key])
            sequencia_max.append(max_diana[sc][key])

        # 3) concatena e guarda
        coluna_media[name_mean] = np.concatenate(sequencia_mean)
        coluna_std   [name_std] = np.concatenate(sequencia_std)
        coluna_max   [name_max] = np.concatenate(sequencia_max)

for sc_label, sc in subcarriers_map.items():
    for esp_id in esps_id:
        # nome da coluna
        name_mean_af = f"Mean sc_{sc_label} esp_{esp_id}"
        name_std_af  = f"Std sc_{sc_label} esp_{esp_id}"
        name_max_af  = f"Max sc_{sc_label} esp_{esp_id}"

        seq_mean, seq_std, seq_max = [], [], []

        for posicao in posicoes_afinar:
            key = f"{posicao}_esp_{esp_id}"

            seq_mean.append(media_afinar[sc][key])
            seq_std.append(std_afinar[sc][key])
            seq_max.append(max_afinar[sc][key])

        coluna_media_afinar[name_mean_af] = np.concatenate(seq_mean)
        coluna_std_afinar[name_std_af] = np.concatenate(seq_std)
        coluna_max_afinar[name_max_af] = np.concatenate(seq_max)

In [ ]:
tamanho_loureiro = len(media_loureiro[1]["vazio_1_esp_1"]) + len(media_loureiro[1]["vazio_1_esp_1"]) + len(media_loureiro[1]["a00_esp_1"]) * 42
print(len(media_loureiro[1]["vazio_1_esp_1"]) + len(media_loureiro[1]["vazio_1_esp_1"]))
print(tamanho_loureiro)

tamanhos_diana = len(media_diana[1]["vazio_1_esp_1"]) + len(media_diana[1]["vazio_1_esp_1"]) + len(media_diana[1]["a00_esp_1"]) * 42
print(len(media_diana[1]["vazio_1_esp_1"]) + len(media_diana[1]["vazio_1_esp_1"]))
print(tamanhos_diana)

total = tamanho_loureiro + tamanhos_diana
print(total)
print(len(coluna_media["Mean sc_1 esp_1"]))

#### Junta e trunca as colunas

Criar a coluna target

In [ ]:
# truncar todas as colunas numa só
# para verificar qual o tamanho do menor vetor
all_cols: dict[str, np.ndarray] = {**coluna_media, **coluna_std, **coluna_max}
min_len: int = min(len(v) for v in all_cols.values())

# trunca as colunas para o tamanho do menor vetor (min_len)
coluna_media = {k: v[:min_len] for k, v in coluna_media.items()}
coluna_std   = {k: v[:min_len] for k, v in coluna_std.items()}
coluna_max   = {k: v[:min_len] for k, v in coluna_max.items()}

# Gera o vetor de labels (0 = vazio, 1 = com_pessoa)
# o número de zeros é a soma dos tamanhos dos grupos "sem pessoa"
# tanto do Loureiro como da Diana
# # a subportadora 22 é escolhida arbitrariamente, pois todas têm o mesmo número de leituras
n_zero: int = sum([
    len(media_loureiro[22][f"vazio_1_esp_{esp_id}"]) +
    len(media_loureiro[22][f"vazio_2_esp_{esp_id}"]) +
    len(media_diana[22][f"vazio_1_esp_{esp_id}"]) +
    len(media_diana[22][f"vazio_2_esp_{esp_id}"])
    for esp_id in esps_id
])

# Total de linhas (após truncar tudo para min_len)
n_total: int = min_len

# Cria vetor target: 0 para "sem pessoa", 1 para "com pessoa"
target_vector: np.ndarray = np.zeros(n_total, dtype = int)
target_vector[n_zero:] = 1  # a partir do índice n_zero, coloca 1 (com pessoa)

#### Dataframe

In [ ]:
# DataFrame
df: pd.DataFrame = pd.DataFrame({**coluna_media, **coluna_std, **coluna_max})
df['target'] = target_vector

print(df['target'].value_counts())
print(df.shape)

# descrever categorical columns
df.describe()

### Machine Learning

#### Divisão treino/teste

In [ ]:
# features
X: np.ndarray = df.drop(columns = 'target').values

# target
y = df['target'].values

print("X.shape =", X.shape)
print("y.shape =", y.shape)


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 200
)

print("X_train.shape =", X_train.shape)
print("X_test.shape  =", X_test.shape)
print("y_train.shape =", y_train.shape)
print("y_test.shape  =", y_test.shape)


#### Random Forest Classifier

In [ ]:
model = RandomForestClassifier(min_samples_split=4,
                                min_samples_leaf=4,
                                max_leaf_nodes=64,
                                n_estimators=150,
                                random_state = 200,
                                max_depth=16,
                                class_weight="balanced_subsample",
                                criterion='entropy'
)

model = model.fit(X_train, y_train)

# pkl_path = path + "\\modelo_treinado.pkl"
# joblib.dump(model, pkl_path)
# print(f"Modelo guardado em: {pkl_path}")

In [ ]:
pred_test = model.predict(X_test)

bal_acc = balanced_accuracy_score(y_test, pred_test)
print("Balanced accuracy:", bal_acc)

#### Confusion Matrix

In [ ]:
confusion_matrix_test = confusion_matrix(y_test, pred_test)

In [ ]:
sns.set(font_scale=1.2)
plt.figure(figsize=(6, 5))

sns.heatmap(confusion_matrix_test, annot=True, fmt='d', cmap='Reds',
            xticklabels=["Sem pessoa", "Com pessoa"],
            yticklabels=["Sem pessoa", "Com pessoa"])

plt.title("Matriz de Confusão")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.tight_layout()
plt.show()


In [ ]:
# pkl_path = path + "\\modelo_treinado.pkl"
# model = joblib.load(pkl_path)

#### Dataframe Afinar

In [ ]:
# verificar menor tamanho das colunas
all_cols_afinar = {**coluna_media_afinar, **coluna_std_afinar, **coluna_max_afinar}
min_len_af: int = min(len(v) for v in all_cols_afinar.values())

# trunca todas as colunas ao mesmo tamanho
coluna_media_afinar = {k: v[:min_len_af] for k, v in coluna_media_afinar.items()}
coluna_std_afinar   = {k: v[:min_len_af] for k, v in coluna_std_afinar.items()}
coluna_max_afinar   = {k: v[:min_len_af] for k, v in coluna_max_afinar.items()}

# Do Miguel:
# # Como há apenas 4 ESPs × 1 ficheiro por classe, a divisão é simples:
# # Cada classe (com_pessoa, vazio) tem 4 séries → total 8
# # O label muda a meio

# Total de linhas após truncamento
n_total_afinar = min_len_af
n_zero_afinar = n_total_afinar // 2  # metade "sem pessoa", metade "com pessoa"

# Gera os rótulos
target_vector_afinar = np.zeros(n_total_afinar, dtype = int)
target_vector_afinar[n_zero_afinar:] = 1

# DF Afinar
df_afinar = pd.DataFrame({**coluna_media_afinar, **coluna_std_afinar, **coluna_max_afinar})
df_afinar['label'] = target_vector_afinar

# Verifica
print(df_afinar['label'].value_counts())
print(df_afinar.shape)
display(df_afinar)

In [ ]:
# Garantir arrays numpy (evita ExtensionArray / typing issues)
X_afinar: np.ndarray = df_afinar.drop(columns='label').to_numpy()
y_afinar: np.ndarray = df_afinar['label'].to_numpy()

# Prever com o modelo atual
y_pred_afinar = model.predict(X_afinar)

# Índices dos erros
falsos_negativos = (y_afinar == 1) & (y_pred_afinar == 0)
falsos_positivos = (y_afinar == 0) & (y_pred_afinar == 1)

# Extrair exemplos incorretamente classificados
X_erros = X_afinar[falsos_negativos | falsos_positivos]
y_erros = y_afinar[falsos_negativos | falsos_positivos]

print(f"Total de exemplos a adicionar (FP + FN): {len(y_erros)}")

# --- Calcular a matriz de confusão ---
cm = confusion_matrix(y_afinar, y_pred_afinar)
print("Confusion Matrix (dataset2):")
print(cm)

In [ ]:

# Reforçar treino com os exemplos mal classificados
X_treinado_reforçado = np.vstack([X_train, X_erros])
y_treinado_reforçado = np.hstack([y_train, y_erros])

print(X_treinado_reforçado.shape, y_treinado_reforçado.shape)
display(X_treinado_reforçado.shape, y_treinado_reforçado.shape)

In [ ]:

modelo2 = RandomForestClassifier(
    min_samples_split=16,
    min_samples_leaf=4,
    max_leaf_nodes=64,
    n_estimators=150,
    random_state=200,
    max_depth=32,
    class_weight="balanced_subsample",
    criterion='entropy',
    n_jobs=-1
)

modelo2.fit(X_treinado_reforçado, y_treinado_reforçado)

y_pred_novo = modelo2.predict(X_test)

print("Balanced Accuracy do novo modelo:", balanced_accuracy_score(y_test, y_pred_novo))

# Guarda o modelo treinado em .pkl (salva modelo2, não model)
# pkl_path = path + "\\modelo_treinado_2.pkl"
# joblib.dump(modelo2, pkl_path)
# print(f"Modelo salvo em: {pkl_path}")

In [ ]:
# Grid Search 
# 432 combinações
param_grid = {
    'min_samples_split': [4, 16, 32],
    'min_samples_leaf': [4, 16, 32, 64],
    'max_leaf_nodes': [4, 16, 32, 64],
    'n_estimators': [100, 150, 200],
    'max_depth': [4, 16, 32]
}

rf = RandomForestClassifier(
    random_state=200,
    class_weight="balanced_subsample",
    criterion='entropy'
)

scorer = make_scorer(balanced_accuracy_score)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
)

# fit
grid_search.fit(X_treinado_reforçado, y_treinado_reforçado)

# Best model and parameters
print("Best balanced accuracy:", grid_search.best_score_)
print("Best params:", grid_search.best_params_)

# Predict with best model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

##### Print:
 * Fitting 5 folds for each of 432 candidates, totalling 2160 fits
 * Best balanced accuracy: 0.9039040507887066
 * Best params: {'max_depth': 16, 'max_leaf_nodes': 64, 'min_samples_leaf': 4, 'min_samples_split': 16, 'n_estimators': 200}
* Test balanced accuracy: 0.9663375492184703